# 03 · Modelado y validación

Se reutiliza la preparación del notebook 02. Los modelos se ajustan con la misma muestra aleatoria de 250.000 filas de entrenamiento (semilla 42) y se comparan en toda la validación. La muestra limita el esfuerzo de entrenamiento. El corte de la etiqueta se calcula sobre todo entrenamiento. Entorno verificado con Python 3.14, pandas 3.1.0, NumPy 2.5.4 y scikit-learn 1.9.1.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

carpeta = next((p for p in (Path("Datasets"), Path("../Datasets")) if p.is_dir()), None)
if carpeta is None:
    raise FileNotFoundError("Abre el notebook desde la raíz de EV3 o desde Notebooks.")

predictoras = ["Grade", "StudyHours", "Extracurricular"]
puntaje = "TestScore_Math"
tipos = {"Grade": "int64", "StudyHours": "float32",
         "Extracurricular": "int64", puntaje: "float64"}

def cargar_particion(nombre):
    datos = pd.read_csv(carpeta / f"{nombre}.csv", usecols=[*predictoras, puntaje],
                        dtype=tipos, float_precision="round_trip")
    validas = (
        datos["Grade"].between(9, 12)
        & np.isfinite(datos["StudyHours"]) & datos["StudyHours"].ge(0)
        & datos["Extracurricular"].isin([0, 1])
        & datos[puntaje].between(0, 100)
    )
    if not validas.all():
        raise ValueError(f"{nombre}: {(~validas).sum():,} filas fuera del esquema esperado")
    return datos

def preparar(datos, corte):
    X = datos[predictoras].astype({"Grade": "int8", "Extracurricular": "int8"})
    y = datos[puntaje].lt(corte).astype("int8").rename("riesgo")
    return X, y

In [2]:
train = cargar_particion("train")
corte_riesgo = float(train[puntaje].quantile(0.20, interpolation="linear"))
X_train, y_train = preparar(train, corte_riesgo)
del train

X_val, y_val = preparar(cargar_particion("validation"), corte_riesgo)
resumen = pd.DataFrame([
    {"partición": nombre, "filas": len(y), "clase positiva": int(y.sum()),
     "proporción positiva": round(float(y.mean()), 6)}
    for nombre, y in [("train", y_train), ("validation", y_val)]
])
print(f"Corte de riesgo: {corte_riesgo!r}")
print(resumen.to_string(index=False))
print("Variables predictoras:", ", ".join(X_train.columns))
assert list(X_train.columns) == list(X_val.columns) == predictoras
assert set(y_train.unique()) == set(y_val.unique()) == {0, 1}

Corte de riesgo: 66.57446853227711
 partición   filas  clase positiva  proporción positiva
     train 8000774         1600155             0.200000
validation  999229          200558             0.200713
Variables predictoras: Grade, StudyHours, Extracurricular


Se elige el mayor *average precision* (AP) en validación. Después se elige el umbral con mayor precisión entre los que alcanzan un recall de al menos 70 %. Ese objetivo exploratorio prioriza detectar casos de bajo rendimiento. La capacidad real de revisión de alertas debe evaluarse antes de usarlo en el colegio.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score, f1_score)

muestra = X_train.sample(n=250_000, random_state=42).index
modelos = {
    "Base": DummyClassifier(strategy="prior"),
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
    "Árbol": DecisionTreeClassifier(max_depth=5, min_samples_leaf=500, random_state=42),
}
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train.loc[muestra], y_train.loc[muestra])
    prob = modelo.predict_proba(X_val)[:, 1]
    resultados.append({"modelo": nombre, "AP": average_precision_score(y_val, prob),
                       "ROC AUC": roc_auc_score(y_val, prob)})
comparacion = pd.DataFrame(resultados).sort_values("AP", ascending=False, kind="stable")
print(comparacion.to_string(index=False))
nombre_final = comparacion.iloc[0]["modelo"]
modelo_final = modelos[nombre_final]
prob_val = modelo_final.predict_proba(X_val)[:, 1]
precision, recall, umbrales = precision_recall_curve(y_val, prob_val)
candidatos = np.flatnonzero(recall[:-1] >= 0.70)
elegido = max(candidatos, key=lambda i: (precision[i], recall[i], umbrales[i]))
umbral_final = float(umbrales[elegido])
print(f"Modelo elegido: {nombre_final}. Umbral: {umbral_final:.8f}")

def resumir(y, prob):
    alerta = prob >= umbral_final
    return {"AP": average_precision_score(y, prob), "ROC AUC": roc_auc_score(y, prob),
            "precisión": precision_score(y, alerta, zero_division=0),
            "recall": recall_score(y, alerta), "F1": f1_score(y, alerta),
            "tasa de alertas": float(alerta.mean()),
            "matriz [[VN, FP], [FN, VP]]": confusion_matrix(y, alerta, labels=[0, 1]).tolist()}

print("Validación:", resumir(y_val, prob_val))

             modelo       AP  ROC AUC
Regresión logística 0.400245 0.726731
              Árbol 0.389098 0.725980
               Base 0.200713 0.500000
Modelo elegido: Regresión logística. Umbral: 0.18694657
Validación: {'AP': 0.40024469115401506, 'ROC AUC': 0.7267314623179791, 'precisión': 0.32151837856406734, 'recall': 0.7000019944355249, 'F1': 0.4406441852581888, 'tasa de alertas': 0.4369869169129399, 'matriz [[VN, FP], [FN, VP]]': [[502412, 296259], [60167, 140391]]}


## Evaluación final

Se aplica el modelo y el umbral fijados en validación. Los falsos negativos son casos de bajo rendimiento omitidos. Los falsos positivos son alertas adicionales que deben revisarse antes de actuar.

In [4]:
X_test, y_test = preparar(cargar_particion("test"), corte_riesgo)
prob_test = modelo_final.predict_proba(X_test)[:, 1]
print("Prueba:", resumir(y_test, prob_test))

Prueba: {'AP': 0.40140234127124397, 'ROC AUC': 0.7279196373989171, 'precisión': 0.3206446489423299, 'recall': 0.7017369876187042, 'F1': 0.44016481125718065, 'tasa de alertas': 0.4369523108569326, 'matriz [[VN, FP], [FN, VP]]': [[503496, 296845], [59550, 140106]]}


## Equidad y explicabilidad

Se muestran métricas agregadas por género, categoría racial y cuartil socioeconómico. El recall por grupo permite comparar la detección de casos positivos. La tasa de alertas mide la carga de revisión y puede variar según la prevalencia. Estas comparaciones describen el dataset de prueba.

Para el modelo logístico se calculan valores de Shapley exactos en escala logit con un fondo marginal de entrenamiento. Para un modelo aditivo, la contribución es el coeficiente multiplicado por la diferencia respecto de la media del fondo, expresada en la escala del modelo. Estas contribuciones explican una salida del modelo y no estiman efectos causales.

In [5]:
grupos = pd.read_csv(carpeta / "test.csv", usecols=["Gender", "Race", "SES_Quartile"])
grupos["real"] = y_test
grupos["alerta"] = prob_test >= umbral_final
grupos["acierto_positivo"] = grupos["real"] * grupos["alerta"]
for atributo in ["Gender", "Race", "SES_Quartile"]:
    tabla = grupos.groupby(atributo).agg(
        n=("real", "size"), positivos=("real", "sum"), alertas=("alerta", "sum"),
        vp=("acierto_positivo", "sum"),
    )
    tabla["recall"] = tabla["vp"] / tabla["positivos"].replace(0, np.nan)
    tabla["tasa_alerta"] = tabla["alertas"] / tabla["n"]
    print(tabla.round(4).to_string()); print()

if nombre_final == "Regresión logística":
    escalador, logistica = modelo_final.steps[0][1], modelo_final.steps[1][1]
    media_fondo = escalador.transform(X_train.loc[muestra]).mean(axis=0)
    referencia = float(logistica.intercept_[0] + media_fondo @ logistica.coef_[0])
    ejemplos = X_test.sample(5, random_state=42)
    aportes = (escalador.transform(ejemplos) - media_fondo) * logistica.coef_[0]
    np.testing.assert_allclose(referencia + aportes.sum(axis=1), modelo_final.decision_function(ejemplos), atol=1e-6)
    detalle = pd.DataFrame(aportes, columns=predictoras)
    detalle["probabilidad"] = modelo_final.predict_proba(ejemplos)[:, 1]
    print(f"Logit base: {referencia:.4f}")
    print(detalle.round(4).to_string(index=False))
    muestra_global = X_test.sample(10_000, random_state=42)
    globales = (escalador.transform(muestra_global) - media_fondo) * logistica.coef_[0]
    print("Importancia global media absoluta:", dict(zip(predictoras, np.abs(globales).mean(axis=0).round(4).tolist())))


             n  positivos  alertas     vp  recall  tasa_alerta
Gender                                                        
Female  510091     101624   222844  71251  0.7011       0.4369
Male    489906      98032   214107  68855  0.7024       0.4370

                  n  positivos  alertas     vp  recall  tasa_alerta
Race                                                               
Asian         49933      10001    21952   7010  0.7009       0.4396
Black        149683      29884    65386  20907  0.6996       0.4368
Hispanic     290332      58077   127095  40823  0.7029       0.4378
Other         20256       4046     8814   2890  0.7143       0.4351
Two-or-more   50079       9981    21620   7010  0.7023       0.4317
White        439714      87667   192084  61466  0.7011       0.4368

                   n  positivos  alertas     vp  recall  tasa_alerta
SES_Quartile                                                        
1             249861      49484   109374  34831  0.7039       0.

## Exportación del modelo para el prototipo

In [6]:
import json

if nombre_final != "Regresión logística":
    raise RuntimeError("El exportador corresponde al modelo logístico evaluado.")
destino = carpeta.parent / "Prototipo" / "modelo.json"
destino.parent.mkdir(exist_ok=True)
parametros = {
    "predictores": predictoras, "media": escalador.mean_.tolist(),
    "escala": escalador.scale_.tolist(), "coeficientes": logistica.coef_[0].tolist(),
    "intercepto": float(logistica.intercept_[0]), "fondo": media_fondo.tolist(),
    "umbral": umbral_final, "corte_rendimiento": corte_riesgo,
}
destino.write_text(json.dumps(parametros, indent=2) + chr(10), encoding="utf-8")
print("Modelo exportado:", destino)

Modelo exportado: Prototipo/modelo.json
